# 19 — Host prompt threading profile

Этот notebook показывает, как пользоваться `HostBatchPolicy(prompt_workers=...)` и `collect_batched_text_rollout_profiled()`.

Цель: понять, простаивает ли GPU потому, что Python последовательно рендерит MegaPrompts / chat templates / `LlmRequest` перед отправкой batch в модель. Здесь LLM заменён scripted backend, чтобы изолированно измерить host-side rollout path. Если на этом notebook `prompt_render_ms` большой и падает при `prompt_workers=4/8`, значит threading действительно помогает именно на подготовке prompts.


In [ ]:
import os

# Keep this first when the same kernel later imports JAX/vLLM/Tunix.
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

print('XLA_PYTHON_CLIENT_PREALLOCATE=', os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'])
print('VLLM_WORKER_MULTIPROC_METHOD=', os.environ['VLLM_WORKER_MULTIPROC_METHOD'])


In [ ]:
from pathlib import Path

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

CONFIG_PATH = ROOT / 'configs/env/text/qwen_craftext.yaml'
OUTPUT_PATH = ROOT / 'artifacts/benchmarks/host-prompt-threading-latest.json'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print('repo:', ROOT)
print('config:', CONFIG_PATH)


In [ ]:
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler

config = load_mvp_config(CONFIG_PATH)
runtime = build_craftext_runtime(config)
task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='fixed',
    fixed_instruction_index=config.environment.instruction_index,
    goal_prefix=config.run.goal,
)
prepared_goal, prepared_instruction_index = task_sampler.task_at(config.environment.instruction_index)
fallback_action_id = runtime.actions.index_of('NOOP')
print('instruction_index:', prepared_instruction_index)
print('goal:', prepared_goal)
print('actions:', runtime.actions.labels)


In [ ]:
import time
from collections.abc import Sequence

from tunix_craftext.env.prompts import MegaPromptRenderer, PromptContext
from tunix_craftext.models.llm import LlmRequest, LlmResponse


class DelayedRenderer:
    """Wrap a real renderer and add controllable per-prompt host work."""

    def __init__(self, renderer, delay_s: float = 0.0) -> None:
        self.renderer = renderer
        self.delay_s = delay_s

    def render(self, context: PromptContext[object]):
        if self.delay_s:
            time.sleep(self.delay_s)
        return self.renderer.render(context)


class ScriptedBatchBackend:
    """Return one valid action per request, preserving batch cardinality and order."""

    def __init__(self, action_label: str = 'NOOP') -> None:
        self.action_label = action_label
        self.calls = 0

    def complete(self, request: LlmRequest) -> LlmResponse:
        return self.complete_batch((request,))[0]

    def complete_batch(self, requests: Sequence[LlmRequest]) -> tuple[LlmResponse, ...]:
        self.calls += 1
        return tuple(
            LlmResponse(f'<action>{self.action_label}</action>', 'scripted', 'host-profile')
            for _ in requests
        )


base_renderer = MegaPromptRenderer(config.prompt.template)
backend = ScriptedBatchBackend(action_label='NOOP')
print('backend:', backend)


## Warmed serial vs threaded prompt render

`delay_s` имитирует дорогой host-side рендеринг: MegaPrompt + chat template + любые Python-преобразования. Важно: первый запуск JAX env часто платит XLA compile cost, поэтому честное сравнение делаем после warmup и несколькими чередующимися повторами. На реальном vLLM notebook можно поставить `delay_s=0.0` и смотреть на настоящий `prompt_render_ms`.


In [ ]:
from statistics import median

from tunix_craftext.rollouts.batched import HostBatchPolicy, collect_batched_text_rollout_profiled


BATCH_SIZE = 8
HORIZON = 4
DELAY_S = 0.01
WORKER_COUNTS = (1, 4)
REPEATS = 3


def run_profile(prompt_workers: int, *, batch_size: int = BATCH_SIZE, horizon: int = HORIZON, delay_s: float = DELAY_S):
    profiled = collect_batched_text_rollout_profiled(
        runtime.adapter,
        DelayedRenderer(base_renderer, delay_s=delay_s),
        backend,
        actions=runtime.actions,
        batch_size=batch_size,
        horizon=horizon,
        seed=config.run.seed,
        goal=prepared_goal,
        max_new_tokens=8,
        invalid_action='fallback',
        fallback_action_id=fallback_action_id,
        host_batch_policy=HostBatchPolicy(prompt_workers=prompt_workers),
    )
    return profiled, profiled.phase_totals_ms()


print('warmup: compiling env reset/step and priming prompt path')
warmup_rollout, warmup_totals = run_profile(prompt_workers=WORKER_COUNTS[-1])
print('warmup totals:', warmup_totals)

runs = []
for repeat in range(REPEATS):
    order = WORKER_COUNTS if repeat % 2 == 0 else tuple(reversed(WORKER_COUNTS))
    for workers in order:
        profiled, totals = run_profile(prompt_workers=workers)
        runs.append({'repeat': repeat, 'prompt_workers': workers, 'totals': totals, 'rollout': profiled})
        print('run', {'repeat': repeat, 'prompt_workers': workers, 'total_ms': totals['total_ms'], 'prompt_render_ms': totals['prompt_render_ms']})

serial_runs = [run for run in runs if run['prompt_workers'] == 1]
threaded_runs = [run for run in runs if run['prompt_workers'] == 4]
serial_totals = {phase: median(run['totals'][phase] for run in serial_runs) for phase in serial_runs[0]['totals']}
threaded_totals = {phase: median(run['totals'][phase] for run in threaded_runs) for phase in threaded_runs[0]['totals']}
serial_min_totals = {phase: min(run['totals'][phase] for run in serial_runs) for phase in serial_runs[0]['totals']}
threaded_min_totals = {phase: min(run['totals'][phase] for run in threaded_runs) for phase in threaded_runs[0]['totals']}
threaded_rollout = threaded_runs[-1]['rollout']


In [ ]:
def ms(value: float) -> str:
    return f'{value:9.2f}'


print('Median phase totals after warmup')
print(f"{'phase':28s} {'serial_ms':>12s} {'threaded_ms':>12s} {'speedup':>10s}")
print('-' * 68)
for phase in serial_totals:
    serial = serial_totals[phase]
    threaded = threaded_totals[phase]
    speedup = serial / threaded if threaded > 0 else float('inf')
    print(f'{phase:28s} {ms(serial)} {ms(threaded)} {speedup:10.2f}x')

print('\nBest observed phase totals after warmup')
print(f"{'phase':28s} {'serial_ms':>12s} {'threaded_ms':>12s} {'speedup':>10s}")
print('-' * 68)
for phase in serial_min_totals:
    serial = serial_min_totals[phase]
    threaded = threaded_min_totals[phase]
    speedup = serial / threaded if threaded > 0 else float('inf')
    print(f'{phase:28s} {ms(serial)} {ms(threaded)} {speedup:10.2f}x')

print('\nActions in threaded rollout step 0:', [a.label for a in threaded_rollout.rollout.decisions[0].actions])


## How to apply this in the real sync vLLM notebook

В `17_sync_vllm_craftext_rollout.ipynb` используйте тот же аргумент:

```python
host_batch_policy=HostBatchPolicy(prompt_workers=4)
```

Если `prompt_render_ms` падает, но `total_ms` почти не меняется, значит bottleneck дальше: lifecycle sync request, decode/dialog bookkeeping, reset/step barriers или слишком мелкие batch/horizon. Если `prompt_render_ms` не падает, renderer/tokenizer не выигрывает от threads — тогда лучше двигаться в async collector или укрупнять request granularity.

Если в первом запуске огромные `environment_step_ms`/`reset_ms`, а после warmup они резко падают, это XLA compile cost. Для performance report сравнивайте warmed median/min, а не первый холодный прогон.


In [ ]:
import json

report = {
    'config_path': str(CONFIG_PATH.relative_to(ROOT)),
    'batch_size': BATCH_SIZE,
    'horizon': HORIZON,
    'delay_s': DELAY_S,
    'repeats': REPEATS,
    'warmup': warmup_totals,
    'median': {'serial': serial_totals, 'threaded': threaded_totals},
    'best': {'serial': serial_min_totals, 'threaded': threaded_min_totals},
    'runs': [
        {'repeat': run['repeat'], 'prompt_workers': run['prompt_workers'], 'totals': run['totals']}
        for run in runs
    ],
    'prompt_workers': {'serial': 1, 'threaded': 4},
}
OUTPUT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True), encoding='utf-8')
print('saved:', OUTPUT_PATH)
